# Revision analysis — PLOS Computational Biology, round 2

This notebook builds the new supplementary figure(s) requested by reviewers, following the same structure as the other notebooks in this repo (R via IRkernel). See project notes for the full reviewer-comment mapping.

Planned panels (may end up as one figure or split into two, decided once results are in):

- **Panel A** — CLIMB component ablation: `props.init` (single-pass) vs `props.corrected` (iterative reweighting) — answers Reviewer 1 W1 (which component drives cross-platform robustness).
- **Panel B** — Reference completeness/coverage sweep — rides along with Reviewer 1 W5 (limitations re: scRNA-seq reference bias in non-hematopoietic contexts). Can reuse `data/simulation_aml/celltype_expr/*_{5,10,20,30,50,100}each.RDS` (already an existing reference-size sweep) rather than building from scratch.
- **Panel C** — CLIFF identifiability under correlated/low-variance cell-subtype proportions across samples — Reviewer 2, comment #1.
- **Panel D** — EM convergence / initialization sensitivity — Reviewer 2, comment #2.
- **Panel E** — Runtime / computational complexity — Reviewer 2, comment #3.
- **Panel F** — Uncertainty propagation from CLIMB proportions into CLIFF sensitivities — Reviewer 2, comment #4.

Starting point per discussion: the **in-vitro cell-line mixture experiment** (`data/invitro_experiment/`) is used first wherever possible, since it has real ground truth for both cell-type proportions (from precise cell counting) *and* cell-type-specific drug sensitivity (dose-response viability measured on the 4 pure cell lines: HL60, K562, SUDHL4, THP1). Panels that need larger sample sizes (C, D, E, F) will extend the existing `Fig3_5_CLIMB_simulation.ipynb` / `Fig6_CLIFF_simulation.ipynb` simulation infrastructure (`data/simulation_aml/`, 200 simulated pseudo-bulks with known ground truth) rather than building a new simulator from scratch.

## Setup

In [ ]:
suppressMessages({
    library(Biobase)
    library(ClimbTheCliff)
    library(glmnet)
    library(ggplot2)
})
set.seed(1)

## Panel A — CLIMB component ablation on in-vitro ground truth (Reviewer 1, W1)

`climb()` returns both `props.init` (first-pass glmnet fit on the full single-cell matrix, before the empirical-Bayes iterative correction) and `props.corrected` (after the iterative reweighting / gene-selection second pass). Comparing the two directly isolates what the iterative-reweighting component contributes, on the 6 in-vitro bulk mixtures with proportions known from precise cell counting.

In [ ]:
sc.es <- readRDS("data/invitro_experiment/invitro_sc_es.RDS")
bulk.es <- readRDS("data/invitro_experiment/invitro_bulk_es.RDS")
true_prop <- as.matrix(read.csv("data/invitro_experiment/true_prop.csv", row.names = 1))
sc.es$cellType <- factor(sc.es$cellType)

climb_out <- climb(sc.es, bulk.es, mode = "abundance", verbose = TRUE)

cn <- colnames(true_prop)
props_init <- as.matrix(climb_out$props.init)[, cn]
props_corrected <- as.matrix(climb_out$props.corrected)[, cn]

rmse <- function(pred, truth) sqrt(mean((pred - truth)^2))
cat("props.init RMSE:     ", round(rmse(props_init, true_prop), 4), "\n")
cat("props.corrected RMSE:", round(rmse(props_corrected, true_prop), 4), "\n")

In [ ]:
# Per-sample RMSE, for a paired before/after plot
per_sample_rmse <- function(pred_mat, truth_mat) {
    sqrt(rowMeans((pred_mat - truth_mat)^2))
}
df_ablation <- data.frame(
    sample = rep(rownames(true_prop), 2),
    rmse = c(per_sample_rmse(props_init, true_prop), per_sample_rmse(props_corrected, true_prop)),
    stage = rep(c("props.init (single-pass)", "props.corrected (iterative)"), each = nrow(true_prop))
)
df_ablation$stage <- factor(df_ablation$stage, levels = c("props.init (single-pass)", "props.corrected (iterative)"))

options(repr.plot.width = 4.5, repr.plot.height = 3.5)
g <- ggplot(df_ablation, aes(x = stage, y = rmse)) +
    geom_boxplot(outlier.shape = NA, width = 0.5) +
    geom_jitter(width = 0.08, height = 0, aes(color = sample), size = 2) +
    theme_classic() +
    ylab("Per-sample proportion RMSE") + xlab(NULL) +
    theme(axis.text.x = element_text(angle = 20, hjust = 1))
g

**Next steps for this panel** (not yet run):
- Extend this same `props.init` vs `props.corrected` comparison across a noise/cross-platform sweep, reusing SFig. 3's 11 bias scenarios and the Fig3 cross-dataset pseudo-bulk pairs (Van Galen↔Naldini, Gray↔Wu, Khaliq↔Lee, Neftel 10x↔SmartSeq2, Jerby-Arnon↔Tirosh), rather than only the single in-vitro noise level shown here.
- Explicitly reframe the existing NNLS-vs-CLIMB comparison (SFig. 9A-D) as the "cell-level resolution" half of this ablation, per the original gap analysis.

## Panel B — Reference completeness/coverage (Reviewer 1, W5)

TODO: reuse `data/simulation_aml/celltype_expr/climb_out_{5,10,20,30,50,100}each.RDS` (already an existing sweep over reference cells-per-type) instead of rebuilding from scratch — need to confirm these were generated with the current `climb()` and check their exact field names before reusing them directly.

## Panels C/D/E/F — Reviewer 2 (identifiability, EM convergence, runtime, uncertainty propagation)

TODO: these need larger sample sizes than the 6 in-vitro mixtures provide. Plan is to extend `data/simulation_aml/` (200 simulated pseudo-bulks, ground-truth proportions + ground-truth cell-type-specific drug sensitivities already exist there) rather than build a new simulation engine, per the shared-simulation-engine design in the revision plan.